# Get to Know a Dataset: The BigBrain

This notebook provides a guided introduction to the **BigBrain** dataset and is intended to accompany its entry in the [Registry of Open Data on AWS](https://registry.opendata.aws/).

The BigBrain is an ultra-high-resolution 3D reconstruction of a complete human brain created from 7,404 coronal histological sections cut at 20 µm, stained for cell bodies, digitized at high resolution, and reconstructed in 3D. The broader BigBrain collection includes volumetric reconstructions, cortical surfaces, classified tissue volumes, cortical layer maps, cytoarchitectonic maps, selected high-resolution histology, and derived datasets.

> **Draft note:** The final Registry landing-page URL, S3 bucket name, AWS region, and example object keys should be inserted once the AWS Open Data bucket and Registry entry are finalized.


### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

The BigBrain collection contains several complementary representations of the same high-resolution histological brain model. The AWS S3 layout should expose these data in a way that makes the major scientific data classes easy to discover.

The expected high-level organization will include categories such as:

1. **3D volumes** — full volumetric reconstructions in histological space and resampled standard spaces, at multiple resolutions.
2. **3D surfaces** — grey- and white-matter surfaces in histological and standard spaces.
3. **3D classified volumes** — tissue classifications such as grey matter, white matter, and CSF.
4. **Cortical layer maps** — derived maps describing cortical laminar structure.
5. **Cytoarchitectonic maps** — histology-derived maps of cortical and subcortical organization.
6. **High-resolution histology** — selected scans at substantially finer resolution.
7. **Derived datasets** — additional BigBrain models, maps, annotations, and analysis products.

The exact S3 prefixes in the code examples below are placeholders until the final AWS bucket layout has been established. The important idea is that users should be able to discover the structure directly from S3 without downloading the full collection.

Additional documentation: https://bigbrainproject.org/maps-and-models.html


In [ ]:
# Required Python packages for this notebook
#
# boto3
# nibabel
# nilearn
# matplotlib
# numpy
#
# Install them with pip or conda using the preferred method for your environment.


First, import the libraries used throughout the notebook. `boto3` provides access to the public S3 bucket, while `nibabel` and `nilearn` are commonly used to work with NIfTI neuroimaging volumes. The visualization example follows the same general approach used in an earlier BigBrain DataLad/CBRAIN tutorial.


In [ ]:
from pathlib import Path
import tempfile

import boto3
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

from botocore import UNSIGNED
from botocore.config import Config
from nilearn import plotting


Define the AWS location of the BigBrain dataset and create an unsigned S3 client. Public AWS Open Data buckets can be queried without AWS credentials.

Replace the placeholder values below once the final BigBrain S3 bucket and AWS region are known.


In [ ]:
# Final values to be inserted after the AWS Open Data bucket is provisioned
bucket = "REPLACE_WITH_BIGBRAIN_S3_BUCKET"
region = "REPLACE_WITH_AWS_REGION"

# Public bucket: do not sign requests
s3 = boto3.client(
    "s3",
    region_name=None if region.startswith("REPLACE_") else region,
    config=Config(signature_version=UNSIGNED),
)

# List top-level objects and prefixes.
response = s3.list_objects_v2(Bucket=bucket, Delimiter="/")

for prefix in response.get("CommonPrefixes", []):
    print("PREFIX:", prefix["Prefix"])

for obj in response.get("Contents", []):
    print("OBJECT:", obj["Key"])


Once the final bucket is populated, inspecting one of the top-level prefixes will reveal the next level of organization. This is useful for exploring the collection before transferring any large files.


In [ ]:
# Replace with one of the actual top-level prefixes returned above.
example_prefix = "REPLACE_WITH_TOP_LEVEL_PREFIX/"

response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix=example_prefix,
    Delimiter="/",
    MaxKeys=100,
)

for prefix in response.get("CommonPrefixes", []):
    print("PREFIX:", prefix["Prefix"])

for obj in response.get("Contents", []):
    print("OBJECT:", obj["Key"])


You can also inspect individual object keys under a selected prefix. This is useful for finding a manageable example file—for example, a downsampled NIfTI volume—without listing the entire multi-terabyte collection.


In [ ]:
# Replace with the actual prefix containing a small or downsampled example dataset.
data_prefix = "REPLACE_WITH_EXAMPLE_DATA_PREFIX/"

response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix=data_prefix,
    MaxKeys=25,
)

for obj in response.get("Contents", []):
    size_mb = obj["Size"] / (1024 ** 2)
    print(f"{obj['Key']}  ({size_mb:.1f} MB)")


### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

The BigBrain collection uses several established neuroscience and 3D geometry formats.

- **NIfTI (`.nii`, `.nii.gz`)** is widely used for volumetric neuroimaging. It stores voxel data together with spatial metadata describing how the image is positioned in physical space. Python users can work with NIfTI using packages such as `nibabel`, `nilearn`, `numpy`, and many neuroimaging toolkits.
- **MINC (`.mnc`)** is another volumetric medical-imaging format used extensively in the BigBrain and McGill neuroimaging ecosystem. MINC supports multidimensional image data and spatial metadata.
- **GIfTI (`.gii`)** is commonly used for neuroimaging surface geometry and associated surface data.
- **OBJ (`.obj`)** is a general 3D geometry format used for cortical and other surface meshes.

BigBrain is available in more than one format because researchers use a wide range of visualization, image-processing, neuroinformatics, and HPC environments. Where possible, the collection exposes equivalent representations so that users can work with the data using established tools.

For introductory Python workflows, a downsampled NIfTI volume is a convenient starting point because it can be loaded directly using `nibabel` and visualized with `nilearn`. For full-resolution data, users should avoid unnecessary transfers and instead work with an appropriate spatial subset, resolution, or compute resource close to the data.


### Q: Can you show us an example of downloading and loading data from your dataset?

The example below downloads a **single manageable NIfTI object** from the public BigBrain S3 bucket to a temporary local directory and opens it using `nibabel`.

The final object key should point to a downsampled BigBrain volume suitable for a short introductory tutorial. The earlier BigBrain tutorial used a 400 µm NIfTI reconstruction in MNI-ICBM152 space for this purpose; the corresponding AWS object key should be substituted once the S3 layout is finalized.


In [ ]:
# Replace with the final key for a manageable NIfTI example file in the AWS bucket.
file_key = "REPLACE_WITH_EXAMPLE_NIFTI_KEY"

tmpdir = Path(tempfile.mkdtemp())
local_file = tmpdir / Path(file_key).name

s3.download_file(bucket, file_key, str(local_file))

img = nib.load(local_file)
print("Loaded:", local_file.name)
print("Shape:", img.shape)
print("Voxel sizes (mm):", img.header.get_zooms()[:3])
print("Data type:", img.get_data_dtype())


The image header tells us the dimensions, voxel spacing, and data type without requiring us to manually interpret the file structure. These metadata are especially important for BigBrain because the dataset is distributed at multiple resolutions and in multiple spatial reference systems.


In [ ]:
print(img.header)


A useful next step is to inspect a spatial subset of the volume. The earlier BigBrain tutorial used `nibabel` slicing together with `nilearn` to explore a cropped portion of the reconstruction rather than rendering an entire large volume at once.


In [ ]:
# Select a central spatial crop.
# The crop is calculated from the actual image dimensions so it remains robust
# if a different downsampled NIfTI example is selected.

shape = np.array(img.shape[:3])
half_width = np.maximum(shape // 8, 1)
center = shape // 2

start = np.maximum(center - half_width, 0)
stop = np.minimum(center + half_width, shape)

cropped = img.slicer[
    start[0]:stop[0],
    start[1]:stop[1],
    start[2]:stop[2],
]

print("Original shape:", img.shape)
print("Cropped shape:", cropped.shape)


### Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset that either illustrates something informative about your dataset, or that you think might excite someone to dig in further.

A defining characteristic of BigBrain is that it connects whole-brain neuroanatomy with microscopic histology. Even a downsampled volume can be explored interactively, while higher-resolution representations support progressively finer inspection of cortical and subcortical structure.

The following visualization adapts the approach used in the earlier BigBrain tutorial: load a NIfTI volume with `nibabel`, select a spatial subset, and render it with `nilearn`.


In [ ]:
# Interactive slice viewer for the cropped BigBrain volume
view = plotting.view_img(
    cropped,
    bg_img=None,
    cmap="gray",
    resampling_interpolation="nearest",
)
view


For a static view, we can also display orthogonal cuts through the downsampled reconstruction.


In [ ]:
plotting.plot_anat(
    img,
    title="BigBrain example volume",
    display_mode="ortho",
)
plt.show()


We can inspect the image intensity distribution as a simple way to understand the numerical range represented in the selected volume. This is not intended as a biological analysis; it is an introductory quality-control step that helps users become familiar with the image values before applying more specialized processing.


In [ ]:
data = np.asarray(cropped.dataobj)
finite_values = data[np.isfinite(data)]

print("Minimum:", finite_values.min())
print("Maximum:", finite_values.max())
print("Mean:", finite_values.mean())
print("Median:", np.median(finite_values))


The histogram below summarizes the voxel intensities in the selected spatial crop.


In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(finite_values.ravel(), bins=100)
plt.title("Voxel intensity distribution in a BigBrain spatial crop")
plt.xlabel("Voxel intensity")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


A key practical lesson is that users do not always need the highest-resolution representation for every task. BigBrain is available at multiple resolutions and in multiple spaces so that researchers can choose an appropriate trade-off between anatomical detail, transfer size, memory use, and computational cost.


In [ ]:
voxel_sizes = np.array(img.header.get_zooms()[:3])
voxel_volume_mm3 = np.prod(voxel_sizes)
n_voxels = np.prod(img.shape[:3])

print(f"Voxel size: {voxel_sizes} mm")
print(f"Voxel volume: {voxel_volume_mm3:.6f} mm³")
print(f"Number of voxels: {n_voxels:,}")
print(f"Uncompressed voxel array size (approx.): {n_voxels * img.get_data_dtype().itemsize / (1024**3):.2f} GiB")


### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

A recurring practical question in BigBrain workflows is:

> **Can we work with BigBrain data without first moving the entire dataset to a local workstation?**

Yes. BigBrain is deliberately distributed through multiple complementary mechanisms, and the AWS Open Data version adds a cloud-native object-access pathway. The introductory workflow above demonstrates the basic pattern:

1. discover the available S3 prefixes and objects;
2. select the resolution and representation appropriate for the analysis;
3. transfer only the object required for the task;
4. load it with standard neuroimaging software; and
5. work with a spatial crop or other restricted representation where appropriate.

This is consistent with earlier BigBrain workflows built around DataLad, Boutiques, CONP, and CBRAIN. DataLad can provide dataset-level versioning and selective retrieval; Boutiques can describe containerized command-line applications in a portable way; and CBRAIN can orchestrate access to data and execution on remote HPC resources. The AWS Open Data distribution adds another complementary access mechanism, particularly useful for cloud-adjacent processing and for platforms such as CBRAIN that can interact with S3-backed datasets.

For larger analyses, the preferred pattern is to **bring compute to the data**, or at least minimize unnecessary transfers, rather than repeatedly copying multi-terabyte collections between sites.


### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

One broad opportunity is to ask how newly developed image-analysis and machine-learning methods can take advantage of BigBrain's multiscale information while remaining computationally efficient and reproducible.

For example:

> **Can automated methods identify or characterize fine-grained cortical and cytoarchitectonic features across very large regions of the BigBrain while preserving anatomical context and remaining practical to execute at scale?**

A useful approach would be to:

1. begin with lower-resolution or spatially restricted BigBrain data for method development;
2. use existing cortical-layer and cytoarchitectonic maps as anatomical context or reference information where appropriate;
3. package analysis tools in containers;
4. describe command-line tools with portable interfaces such as Boutiques when suitable;
5. scale execution using HPC or cloud resources rather than relying on a single workstation; and
6. record software versions, parameters, spatial references, and provenance so results can be reproduced.

The AWS Open Data distribution is intended to make this type of scalable experimentation easier by placing an openly accessible copy of the BigBrain collection in object storage that can be reached directly from cloud and federated computational environments.

Additional BigBrain tools and tutorials are available through:
- https://bigbrainproject.org/tools-and-services.html
- https://siibra-python.readthedocs.io/
- https://bigbrainwarp.readthedocs.io/
- https://www.cbrain.ca/


# Before publishing

Before committing this notebook to the GitHub repository:

1. Replace all `REPLACE_WITH_...` placeholders with the final AWS bucket, region, prefixes, and example object key.
2. Confirm that the selected example NIfTI object is small enough for an introductory notebook.
3. Run the notebook from a clean Python environment and verify that all cells execute successfully.
4. Confirm that public S3 access works without AWS credentials.
5. Clear all notebook outputs before committing the final version, unless AWS specifically requests retained example outputs.
6. Update the Registry landing-page link in the introduction once the final BigBrain Registry slug is confirmed.
